In [ ]:
from typing import List, TypedDict
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from dotenv import load_dotenv

load.dotenv()


In [ ]:
# docs loading
docs = (
    PyPDFLoader("./documents/Project\RAG_Campus_X\Advance_RAG (CRAG)/book1.pdf").load()
    + PyPDFLoader("./documents/Project\RAG_Campus_X\Advance_RAG (CRAG)/book2.pdf").load()
    + PyPDFLoader("./documents/Project\RAG_Campus_X\Advance_RAG (CRAG)/book3.pdf").load()
)

In [ ]:
# text splitting
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)
for d in chunks:
  d.page_content = d.page_content.encode("utf-8", 'ignore').decode("utf-8", 'ignore')

In [ ]:
# embedding --> vectore store and created retriever

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
# craeating llm model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
UPPER_TH = 0.7
LOWER_TH = 0.3

In [ ]:
# defining states (this is a langgraph process)

class State(TypedDict):
  question:str
  docs:List[Document]

  good_docs: List[Document]     # means those docs which has a evaluation greater than 0.3
  verdict: str                  # here we will store that three values correct, ambicious, incorrect
  reason: str                   # here we will store the reason of each verdict

  strips:List[str]          # output of decomposition (sentence strips)
  kept_strips:List[str]     # after filtering (kept sentences)
  refined_context:str       #recomposed internal knowledge (joined kept_strips)

  web_docs: List[Document]   # used to store a doc whihc we will get through web
  web_query: str             # when we will web_search then our query should be more understandable that's why.

  answer:str

In [ ]:
# retrieved node

def retrieved(state:State) -> State:
  q = state['question']
  return {'docs': retriever.invoke(q)}

In [ ]:
#-----------------------
# score based doc evaluator
#-----------------------

class DocEvalScore(BaseModel):
  score:float
  reason:str

doc_eval_propmt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.1, 1.0].\n"
            "- 1.0 : chunk alone is sufficinet to answer fully/mostly\n"
            "- 0.0 : chunk is relevant"
            "Be conservative with high scores.\n"
            "Also return a short answer.\n"
            "Ouput JSON only.\n"
        ),
        ("human","Question: {question}\n\nChunk:\n{chunk}"),
    ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

In [ ]:
# This func will perform evaluation (easy)
def eval_each_dc_node(state:State) -> state:

  q = state('question')     # fetching ques

  scores: List[float] = []
  reasons: List[str]= []
  good: List[Document] = []    # creating empty lists

  for d in state['docs']:    # loop chala rhe hai
    out = doc_eval_chain.invoke({"question":q, "chunk":d.page_content})   # provind llm a ques and chunk
    scores.append(out.score)
    reasons.append(out.reason)   # and asking it a score ans reason

    # 5) for CORRECT case we will refine only docs with score > LOWER.TH
    if out.score > LOWER_TH:
      good.append(d)

  # 2) CORRECT if atleast one doc > UPPER_TH
  if any(s > UPPER_TH for s in scores):
    return {
        "good_docs":good,
        "verdict":CORRECT,
        "reason":f"At least one retrieved chunk scored > {UPPER_TH}.",
          }

  # 3) INCORRECT if all docs < LOWER_TH
  if len(scores) > 0 and all(s < LOWER_TH for s in scores):
    why = "No chunk is sufficient "
    return {
        "good_docs":[],
        "verdict":INCORRECT,
        "reason":f"All retrieved chunk scored < {LOWER_TH}. {why}",
          }

  # 4) Anything inbetween => AMBIGUOUS
  why = "Mixed relevence signals."
  return {
      "good_docs":[],
      "verdict":AMBIGUOUS,
      "reason":f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}. {why}",
        }




In [ ]:
# this will be a refinement node

#---------------------
# Sentence-level Decomposition
#---------------------


# It will convert the given str into strips
def decompose_to_sentences(text: str) -> List[str]:
  text = re.sub(r"\s+", " ", text).strip()
  sentences = re.split(r"(?<=[.!?])\s+", text)
  return [s.strip() for s in sentences if len(s.strip()) > 20]


#---------------------
# Filter LLM(judge)
#---------------------

#Giving prompt to llm

class KeepOrDrop(BaseModel):
  keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
     [
         (
             "system",
             "You are a strict relevance filter.\n"
             "Return keep=true only if sentence directly helps answer the question.\n"
             "Use only the sentence. Output JSON only."
         ),
         ("human", "Question: {question}\n\nSentence: {sentence}"),
     ]
 )

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

#-----------------------
# REFINING (Decompose -> Filter -> Recompose)
#-----------------------

# this will be a refining node

def refine(state:State) -> State:

  q = state['question']

 # Combine retrieved doc into one context string
if state.get("verdict") == 'CORRECT':
  context = "\n\n".join(d.page_content for d in State['good_docs']).strip()
 else:
  context = "\n\n".join(d.page_content for d in State['web_docs']).strip()


# 1) DECOMPOSITION : context -> sentence strips
strips = decompose_to_sentences(context)      # we are creating s strips here

# 2) FILTER: keep only relevant strips
kept: List[str] = []

for s in strips:
  if filter_chain.invoke({"question": q, "sentence": s})
      kept.append(s)

      # what we are doing here , we are picking every strips and sending it to filter_chain.
      # Then it will give que and strips to llm and asking llm that "is it right strip for that
      # ques and if yes then it adding to that kept wali line."

# 3) RECOMPOSE : glue kept strip back together (internal knowledge)

refined_context = "\n".join(kept).strip()    # kept strips storing in this variable

return {
    "strips": strips,
    "kept_strips": kept,
    "refined_context": refined_context

}

In [ ]:
class WebQuery(BaseModel):
  query: str

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords.\n"
            "Rules.\n"
            "-Keep it shorty(8-14 words).\n"
            "- if the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
            "-Do not answer the queestion.\n"
            "-Return JSON with a single key:query",
        ),
        ("human", "question:" {question}),
    ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)

def rewrite_query_node(state:State) -> State:
  out = rewrite_chain.invoke({"question":state["question"]})
  return {"web_query":out.query}

  # why there is a need of thid funs?
  # => In websearch we are giving that query to web. Inbetween this there is a chances a query might get meaningless thus here we are maintaining a meaning of that query as it is.




#----------------------
#Web searach (Iteration 4)
#Assumption : web search does not fail (not fail node in this branch )
#----------------------

tavily = TavilySearchResult(max_results=5)

def web_seach_node(state:State) -> State:
  q = state.get("web_query") or state['question']               # no query rewrite
  results = tavily.invoke({"query": q})     # no knowledge selection

  web_docs = []
  for r in results or []

     title = r.get("title", "")
     url = r.get("url", "")
     content = r.get("content", "") or r.get("snippet", "")

     text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"

     web_docs.append(Document(page_content=text meatdata={'url':url, 'title:'title}))

  return {"web_docs": web_docs}


In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer only using the provided refined bullets.\n"
            "If the bullets are empty or insufficient, say: 'I don't know based on provided books.'",
        ),
    ]
)

def genrate(state:State) -> State:
  out = (answer_prompt | llm).invoke({"question": state['question'],"refined_context": state['refined_context']})
  return {"answer": out.content}

In [ ]:
def ambiguous_node(state:State) -> State:
  return {"answer": f"Ambiguous: {state['reason']}."}

def route_after_eval(state:State) -> State:
  if state['verdict'] == CORRECT:
    return "refine"
  elif state['verdict'] == INCORRECT:
    return "web_search"
  else:
    return "ambiguous"

In [ ]:
g = StateGraph(State)
g.add_node("retrive", retrive_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("refine", refine)
g.add_node("genrate", genrate)
g.add_node("fail", fail_node)
g.add_node("ambiguous", ambiguous_node)

g.add_edge(START, "retrive")
g.add_edge("retrive", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {"refine":refine, "web_search": "fail", "ambiguous":"ambiguous"}
)
g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search","refine")     # INCORRECT -> refine(web_docs)
g.add_edge("refine", "genrate")       # CORRECT/INCORRECT -> generate
g.add_edge("genrate", END)
g.add_edge("ambiguous", END)


app = g.compile()
app

# here we are joining these three things 1)retrive, 2)refine, 3) generate

In [ ]:
res = app.invoke({
    "question": "What is nykaa advertising about?",
    "docs":[],
    "good_docs": [],
    "verdict":[],
    "reason":[],
    "strips":[],
    "kept_strips":[],
    "refined_context":"",
    "web_docs": [],
    "answer":"",
})
print("VERDICT:", res['verdict'])
print("REASON:", res['reason'])
print("\nOUTPUT:\n", res['answer'])      #


Tll here we have completed with all the steps.
1) Refined the meaning full content from the pdf
2) differentaite it into correct , incorrect or ambiguous depennds upn a provided query.
3) if used websearch if we found a incorrect case.